In [ ]:
import ctypes
import numpy as np
import time
import cv2
import gc
import os
import zlib
from sdlarch_rl import make
import pygame
import gzip
import ctypes

# env = make("SuperStreetFighterIV-3DS", env_variables=[{ "citra_layout_option": "Default Top-Bottom Screen"}])
env = make("SuperStreetFighterIV-3DS")
# env = make("GranTurismo3-Ps2")
#env = make("NewSuperMarioBros-Wii", env_id=1)
# env = make("VirtuaTennis-DC")
# env = make("CrazyTaxi-DC")
# env = make("SuperMario64-N64")
# env = make("SuperSmashBrosBrawl-Wii")
# env = make("GodOfWar-PSP")
# env = make("YoshiIsland-NDS")

obs, info = env.reset()

count = 0
global initial_state
initial_state = None

pygame.init()

pygame.joystick.init()
joysticks = [pygame.joystick.Joystick(x) for x in range(pygame.joystick.get_count())]

print("joysticks count: ", len(joysticks))

buttons = None
if len(joysticks) > 0:
    joystick = joysticks[0]
    name = joystick.get_name()
    print(name)
    buttons = joystick.get_numbuttons()
    # for i in range(buttons):
    #     button = joystick.get_button(i)
    #     print(f"Button {i:>2} value: {button} ")

    # hats = joystick.get_numhats()
    # print(f"Number of hats: {hats}")

    # for i in range(hats):
    #     hat = joystick.get_hat(i)
    #     print(f"Hat {i} value: {str(hat)}")

SCREEN_WIDTH = 640
SCREEN_HEIGHT = 480

if os.name == 'nt':
    SCREEN_WIDTH = 768*3
    SCREEN_HEIGHT = 640*3
    
window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

paused = False

# Wii games
INVERT_AXIS=False

framerate = env.unwrapped.em.get_frame_rate()

print("framerate: ", framerate)

frame_time = 1.0 / framerate
last_time = time.time()

def get_ptr():
    capsule = env.unwrapped.em.get_memory_pointer()
    PyCapsule_GetPointer = ctypes.pythonapi.PyCapsule_GetPointer
    PyCapsule_GetPointer.restype = ctypes.c_void_p
    PyCapsule_GetPointer.argtypes = [ctypes.py_object, ctypes.c_char_p]
    ptr = PyCapsule_GetPointer(capsule, None)
    return ptr

ptr = get_ptr();
print("Pointer address:", hex(ptr))

while True:
    pygame.event.pump()
        
    keys = pygame.key.get_pressed()

    # Pause
    if keys[pygame.K_BACKSPACE]:
        paused = not paused
        print("== PAUSED ==" if paused else "== RETURNING ==")
        time.sleep(0.3)  # debounce

    if keys[pygame.K_s]:
        paused = not paused

        if paused:
            state = env.unwrapped.em.get_state()
            with gzip.open("default.state", "wb") as f:
                f.write(state)

    if paused:
        clock.tick(10)
        continue

    # Xbox 360
    # A Button        - Button 0
    # B Button        - Button 1
    # X Button        - Button 2
    # Y Button        - Button 3
    # Left Bumper     - Button 4
    # Right Bumper    - Button 5
    # Back Button     - Button 6
    # Start Button    - Button 7
    # L. Stick In     - Button 8
    # R. Stick In     - Button 9
    # Guide Button    - Button 10

    # libretro
    # ["B", "Y", "SELECT", "START", "UP", "DOWN", "LEFT", "RIGHT", "A", "X", "L1", "R1", "L2", "R2", "L3", "R3"]
    
    action = np.zeros(16, dtype=np.uint8)

    for i in range(buttons):
        button = joystick.get_button(i)

        if button > 0:
            if i == 0:
                action[8] = button
            if i == 1:
                action[0] = button
            if i == 2:
                action[9] = button
            if i == 3:
                action[1] = button
            if i == 8:
                action[14] = button
            if i == 9:
                action[15] = button
            if i == 7:
                action[3] = button
    hat_x, hat_y = joystick.get_hat(0)

    if hat_y == 1:
        action[4] = 1
    elif hat_y == -1:
        action[5] = 1
    if hat_x == -1:
        action[6] = 1
    elif hat_x == 1:
        action[7] = 1

    left_x  = joystick.get_axis(0)
    left_y  = joystick.get_axis(1)
    right_x = joystick.get_axis(2)
    right_y = joystick.get_axis(3)
    lt      = joystick.get_axis(4)
    rt      = joystick.get_axis(5)

    if left_y < -0.5:
        action[4] = 1
    elif left_y > 0.5:
        action[5] = 1

    if left_x < -0.5:
        action[6] = 1
    elif left_x > 0.5:
        action[7] = 1

    if keys[pygame.K_UP]:
        if INVERT_AXIS:
            action[7] = 1
        else:
            action[4] = 1
    if keys[pygame.K_DOWN]:
        if INVERT_AXIS:
            action[6] = 1
        else:
            action[5] = 1
    if keys[pygame.K_LEFT]:
        if INVERT_AXIS:
            action[4] = 1
        else:
            action[6] = 1
    if keys[pygame.K_RIGHT]:
        if INVERT_AXIS:
            action[5] = 1
        else:
            action[7] = 1
    if keys[pygame.K_c]:
        action[1] = 1
    if keys[pygame.K_x]:
        action[0] = 1
    if keys[pygame.K_RETURN]:
        action[3] = 1
    if keys[pygame.K_l]:
        action[11] = 1

    # clock.tick(60)
    
    img, rew, done, _, info = env.step(action)

    n_ptr = get_ptr()

    if ptr != n_ptr:
        print("new ptr:", hex(n_ptr))
        ptr = n_ptr

    # ram = env.unwrapped.em.get_ram()
    # offset=0x6EA38B0
    # size=1
    # t="<u1"
    # val = int(np.frombuffer(ram[offset:offset + size], dtype=t)[0])
    # print("Life:", val)
    print(info)

    # print("Raw bytes:", [hex(x) for x in ram[offset:offset+8]])
    
    # for dtype in ["<u1", "<u2", "<u4", "<i4", ">u4"]:
    #     val = np.frombuffer(ram[offset:offset+4], dtype=dtype)[0]
    #     print(dtype, "=", val)
    # print(ram[0x6EA38B0])
    # base = 0x08000000
    # offset = addr - base
    
    # print([hex(b) for b in ram[offset:offset+4]])
    # value = int.from_bytes(ram[offset:offset+4], byteorder='little')
    # print(value)

    font = cv2.FONT_HERSHEY_SIMPLEX
            
    # org
    org = (400, 50)
    
    # fontScale
    fontScale = 1
     
    # Blue color in BGR
    color = (255, 0, 0)
    
    # Line thickness of 2 px
    thickness = 2

    img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
     
    # Using cv2.putText() method
    img = cv2.putText(img, "Paused: " + str(paused + 1), org, font, 
                       fontScale, color, thickness, cv2.LINE_AA)


    surface = pygame.surfarray.make_surface(np.transpose(img, (1, 0, 2)))


    window.blit(surface, (0, 0))
    pygame.display.update()

    count += 1

    clock.tick(60)

    # break

    if count % 1000 == 0:
        # env.reset()
        pass

    

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state
joysticks count:  1
Xbox 360 Controller
framerate:  60.0
Pointer address: 0x1b7e0830040
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'time': 99.0, 'player1': 128.0, 'player2': 128.0}
{'